## Run Embeddings on images

1. Read in all images
2. YOLOv8 detect on all images
3. YOLOv8 embeddings on all images
4. YOLOv8 embeddings on all crops of images
5. PyTorch Face Detect and embeddings
6. Database all detections from YOLOv8 and Faces
7. Database all embedding vectors, Full, Crops, Face
8. Search Vector db for similar images, objects, faces
9. Add labels and tags to vectors/postgres for querying quickly

In [5]:
from ultralytics import SAM

# Load a model
model = SAM('sam_b.pt')

# Display model information (optional)
model.info()

# Run inference with bboxes prompt
model(image_file, bboxes=[439, 437, 524, 709])

# Run inference with points prompt
model(image_file, points=[900, 370], labels=[1])

Model summary: 238 layers, 93735472 parameters, 93735472 gradients

image 1/1 /home/superdave/Pictures/david_cindy.jpg: 1024x1024 147.3ms
Speed: 8.3ms preprocess, 147.3ms inference, 0.5ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/superdave/Pictures/david_cindy.jpg: 1024x1024 136.0ms
Speed: 5.0ms preprocess, 136.0ms inference, 0.3ms postprocess per image at shape (1, 3, 1024, 1024)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: None
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: '0'}
 orig_img: array([[[145, 145, 145],
         [141, 143, 143],
         [145, 147, 147],
         ...,
         [225, 238, 236],
         [224, 238, 234],
         [223, 237, 233]],
 
        [[149, 149, 149],
         [145, 147, 147],
         [146, 151, 150],
         ...,
         [225, 238, 236],
         [224, 238, 234],
         [224, 238, 234]],
 
        [[151, 153, 153],
         [149, 151, 151],
         [149, 154, 153],
         ...,
         [226, 239, 237],
         [225, 239, 235],
         [225, 239, 235]],
 
        ...,
 
        [[ 14,   6,   0],
         [ 15,   7,   0],
         [ 15,   7,   0],
         ...,
         [ 95,  90,  89],
         [ 98,  90,  90],
         [ 98,  89,  86]],
 
        [[ 15,   7,   0],
         [ 15,   7,   0],
         [ 16,   8,   1],
         ...,
         [ 96,  93,  88],
        

In [3]:
from ultralytics import NAS

# Load a COCO-pretrained YOLO-NAS-s model
model = NAS('yolo_nas_s.pt')

# Display model information (optional)
model.info()

# Validate the model on the COCO8 example dataset
# results = model.val(data='coco8.yaml')

image_file = '/home/superdave/Pictures/david_cindy.jpg'
# Run inference with the YOLO-NAS-s model on the 'bus.jpg' image
results = model(image_file)

Model summary: 685 layers, 19053888 parameters, 19053888 gradients



RuntimeError: Inference tensors do not track version counter.

In [1]:
import torch

if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        print(f"Index: {idx}, GPU: {torch.cuda.get_device_name(idx)}")
else:
    print("No GPUs available.")


Index: 0, GPU: NVIDIA RTX A6000
Index: 1, GPU: NVIDIA RTX A6000
Index: 2, GPU: Quadro M5000


In [3]:
from utils import get_files, image_resize, get_max_dimension, pad_batch_images, parallel_convert_coordinates, draw_image_with_all_boxes, extract_faces_from_data, normalize_to_L2 
from load_models import load_all_modes
from detect import results_to_dataframe_img_crops, get_embeddings, model_embedding_norm
from database_sql import insert_metadata_bulk, insert_detections_bulk, create_tables, clear_and_delete_all_data
from database_vec import create_new_collection, milvus_similarity_search, milvus_radius_search
from database_sql_model import ImageData

from concurrent.futures import ThreadPoolExecutor
from itertools import repeat
from gc import collect
from pymilvus import utility
import numpy as np
import pandas as pd
from dataclasses import asdict

Connecting to Milvus
Milvus connection established


/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
reg_model, emb_model, face_mtcnn, face_resnet, device_loc = load_all_modes(0)

Loading the Models


In [5]:
device_loc

device(type='cuda', index=0)

## Get all images

In [6]:
test_dir = '/mnt/nas/test_emb_imgs/'
image_files = get_files(test_dir)
len(image_files)

74

In [18]:
image_files = ['/home/superdave/Pictures/david_cindy.jpg']

## Get detections of all Images

#### Resize Images for Speed

In [19]:
with ThreadPoolExecutor() as executer:
    resized_images = list(executer.map(image_resize, image_files, repeat(640)))
len(resized_images)

1

In [20]:
resized_images_pad_bgr = pad_batch_images(resized_images)

#### Detect

In [ ]:
results = reg_model.predict(resized_images, imgsz = 640, verbose = True, stream=False, device=device_loc) 

## Detection Processing

In [ ]:
img_metadata, img_detects, img_crops = results_to_dataframe_img_crops(results, image_files, reg_model)

## Complete Embeddings Full & Crop

### Crops

In [ ]:
# using image crops do the embeddings
if len(img_crops) > 0:   
    image_crop_embeddings_yolov8 = get_embeddings(img_crops, emb_model, device=device_loc)

### Full Image

In [15]:
resized_images_pad_bgr[0].shape

(640, 640, 3)

In [16]:
type(resized_images_pad_bgr[0])

numpy.ndarray

In [21]:
embeddings_batch = model_embedding_norm(emb_model, resized_images_pad_bgr, 'cuda:0')

In [22]:
embeddings_batch

[[-0.011015619151294231,
  -0.007119582500308752,
  -0.006394857540726662,
  0.017972007393836975,
  -0.04037985950708389,
  -0.05063879117369652,
  0.06309368461370468,
  0.012595470063388348,
  0.038693927228450775,
  -0.04161274433135986,
  0.05781619995832443,
  -0.02451193705201149,
  0.0175789762288332,
  0.08905266970396042,
  -0.0024375536013394594,
  0.028803935274481773,
  0.15851394832134247,
  -0.04498874768614769,
  0.07383424043655396,
  0.050496943295001984,
  0.006143838632851839,
  -0.044488903135061264,
  0.038913968950510025,
  -0.020005811005830765,
  -0.00011857067875098437,
  0.01786748692393303,
  -0.023084454238414764,
  -0.001364972209557891,
  -0.02611072175204754,
  -0.03498227149248123,
  -0.016519371420145035,
  0.004734588321298361,
  -0.05033733323216438,
  -0.015723463147878647,
  0.06900284439325333,
  0.07703672349452972,
  -0.02329838275909424,
  0.006103163585066795,
  -0.03613293915987015,
  0.01694921776652336,
  0.11211959272623062,
  -0.025463568

# Database

## Postges

#### Clear Database

In [ ]:
clear_and_delete_all_data()

#### Create Tables

In [ ]:
create_tables()

#### Insert Metadata & Detections

In [ ]:
print('Database to SQL the detections and metadata')
# database the dataframes
insert_metadata_bulk(img_metadata)
insert_detections_bulk(img_detects)

## Milvus

#### Create Collections

In [ ]:
utility.list_collections()

In [ ]:
for item in utility.list_collections():
    utility.drop_collection(item)

In [ ]:
new_collection = "detections"
utility.drop_collection(new_collection)
detections_collection = create_new_collection(new_collection)

In [ ]:
# create Full Image Collection
new_collection = "full_image"
utility.drop_collection(new_collection)
fullimg_collection = create_new_collection(new_collection)

In [ ]:
# create Full Image Collection
new_collection = "facial"
utility.drop_collection(new_collection)
facial_collection = create_new_collection(new_collection, 512)

In [ ]:
utility.list_collections()

#### Insert Embeddings

In [ ]:
# write the crop embeddings to milvus
detections_collection.insert([img_detects['uuid'].tolist(), img_detects['file_uuid'].tolist(), image_crop_embeddings_yolov8, img_detects['class_name'].tolist(), list(repeat('none', len(img_detects)))])

In [ ]:
# write the full embeddings to milvus
fullimg_collection.insert([img_metadata['file_uuid'].tolist(), image_files, image_full_embeddings_yolov8, list(repeat('none', len(img_metadata['file_uuid']))), list(repeat('none', len(img_metadata['file_uuid'])))])

## Clean up memory

In [ ]:
del results, image_crop_embeddings_yolov8, image_full_embeddings_yolov8
collect()
torch.cuda.empty_cache()

## Query with Image

In [ ]:
test_img = '/mnt/nas/test_emb_imgs/IMG_4110.JPG'

### Embed Image

In [ ]:
resized_single = image_resize(test_img, 640)

In [ ]:
single_embedding = get_embeddings(resized_single, emb_model, device=device)[0]

In [ ]:
# must load collection to search
fullimg_collection.load()

In [ ]:
search_results = milvus_similarity_search(fullimg_collection, single_embedding, 10)

In [ ]:
for hits in search_results:
    # get the distances to the query vector from all returned hits
    for hit, dis in zip(hits, hits.distances):
        # get the value of an output field specified in the search request.
        # dynamic fields are supported, but vector fields are not supported yet.
        print(hit.entity.get('uuid'))
        print(hit.entity.get('img_path'))
        print(dis)

In [ ]:
# fullimg_collection.release()

#### Search Radius Option

In [ ]:
res = milvus_radius_search(fullimg_collection, single_embedding)

In [ ]:
for hits in res:
    # get the distances to the query vector from all returned hits
    for hit, dis in zip(hits, hits.distances):
        # get the value of an output field specified in the search request.
        # dynamic fields are supported, but vector fields are not supported yet.
        print(hit.entity.get('uuid'))
        print(hit.entity.get('img_path'))
        print(dis)

## Read HEIC Images

requires linux packages: apt install libffi libheif-dev libde265-dev

In [ ]:
import pyheif
from PIL import Image

In [ ]:
test_img = '/mnt/nas/Pictures/2020-10-18/IMG_8435.HEIC'

In [ ]:
heif_file = pyheif.read(test_img)

In [ ]:
heif_file

In [ ]:
image = Image.frombytes(
    heif_file.mode, 
    heif_file.size, 
    heif_file.data,
    "raw",
    heif_file.mode,
    heif_file.stride,
    )

In [ ]:
type(image)

In [ ]:
from numpy import array

In [ ]:
opencv_image = array(image)

In [ ]:
opencv_image

## Open RAW cr2 files

In [ ]:
import rawpy
from cv2 import cvtColor, COLOR_RGB2BGR, imwrite, imread
from os import path

In [ ]:
raw_img = '/mnt/nas/Pictures/2020-07-26/IMG_6640.CR2'
# raw_img = '/mnt/nas/Pictures/2019-11-23/IMG_3828.CR2'

In [ ]:
with rawpy.imread(raw_img) as raw:
    rgb = raw.postprocess(use_camera_wb=True)
    bgr = cvtColor(rgb, COLOR_RGB2BGR)
    imwrite("image.jpg",bgr)

## Facenet PyTorch

In [ ]:
max_size = get_max_dimension(resized_images)

In [ ]:
resized_face_imgs = pad_batch_images(resized_images, im_size = max_size)

#### Detect faces in images

In [ ]:
faces_aligned, probs = face_mtcnn.detect(resized_face_imgs)

##### Sort through detect results

In [ ]:
# Construct list of ImageData instances
image_data_list = [
    ImageData(
        file, 
        item, 
        org_uuid, 
        p, 
        resized_images[i].shape[:2], # Assuming the shape is in the form (height, width, channels)
        resized_face_imgs[i].shape[:2] # Assuming the shape is in the form (height, width, channels)
    )
    for i, (file, box, prob, org_uuid) in enumerate(zip(image_files, faces_aligned, probs, img_metadata['file_uuid'].tolist()))
    if isinstance(box, np.ndarray)
    for item, p in zip(box, prob)
]

### Convert boxes to Normalized for Databasing

In [ ]:
# Generate the records using parallel processing
df_faces = parallel_convert_coordinates(image_data_list)
df_faces

### Database the Dataframe to Postgres

In [ ]:
db_df_faces = df_faces.drop(columns=['path'])
insert_detections_bulk(db_df_faces)

### Draw on Original image to check for accuracy

In [ ]:
# Call the function
draw_image_with_all_boxes(df_faces, img_metadata)

#### Functions to resize and crop from Face Detections

In [ ]:
extracted_faces = extract_faces_from_data(df_faces, device)

### Get Embeddings of Faces found

In [ ]:
embeddings = face_resnet(extracted_faces)
# embeddings are 512 long

In [ ]:
vector_list = [normalize_to_L2(tensor.tolist()).tolist() for tensor in embeddings]

In [ ]:
len(vector_list)

### Database Embeddings to Milvus

In [ ]:
# write the crop embeddings to milvus
facial_collection.insert([df_faces['uuid'].tolist(), df_faces['path'].tolist(), vector_list, list(repeat('face', len(df_faces))), list(repeat('none', len(df_faces)))])

In [ ]:
del resized_face_imgs, faces_aligned, probs, embeddings
collect()
torch.cuda.empty_cache()

## TODO: Right a function to extract face in image, then query faces collection

return the uuids of the faces to retrieve the file uuids for the image retrieval

In [ ]:
face_img = '/home/superdave/Pictures/david_crop.JPG'

In [ ]:
def get_single_img_embedding(image_path):
    
    # resize input image
    resized_face = image_resize(image_path, 640)
    
    # pad image as required
    max_size = get_max_dimension(resized_face)
    resized_face_imgs = pad_batch_images(resized_face, im_size = max_size)
    
    # detect face
    faces_aligned, probs = face_mtcnn.detect(resized_face_imgs)
    
    # organize results
    image_data_list = [
        ImageData(
            file, 
            item, 
            org_uuid, 
            p, 
            resized_face.shape[:2], # Assuming the shape is in the form (height, width, channels)
            resized_face_imgs[i].shape[:2] # Assuming the shape is in the form (height, width, channels)
        )
        for i, (file, box, prob, org_uuid) in enumerate(zip([face_img], faces_aligned, probs, repeat(None)))
        if isinstance(box, np.ndarray)
        for item, p in zip(box, prob)
        ]
    
    records = parallel_convert_coordinates(image_data_list)
    
    # crop to face of interest
    extracted_face_single = extract_faces_from_data(records, device_loc)
    
    # get embedding of face and normalize
    embedding_single = face_resnet(extracted_face_single)
    search_vector = normalize_to_L2(embedding_single[0].detach().cpu().numpy())
    
    return search_vector

In [ ]:
# tensor  = extracted_face_single[0].cpu().numpy() # make sure tensor is on cpu
# image_np = np.transpose(tensor, (1, 2, 0))
# imwrite("face_crop_test.jpg", image_np)

In [ ]:
search_vector = get_single_img_embedding(face_img)

### Query the Face Vectors

In [ ]:
from pymilvus import Collection

In [ ]:
collection_face = Collection('facial')
collection_face.load()

In [ ]:
res = milvus_radius_search(collection_face, search_vector, )

In [ ]:
for hits in res:
    # get the distances to the query vector from all returned hits
    for hit, dis in zip(hits, hits.distances):
        # get the value of an output field specified in the search request.
        # dynamic fields are supported, but vector fields are not supported yet.
        print(hit.entity.get('img_path'))
        print(dis)

In [ ]:
search_results = milvus_similarity_search(collection_face, search_vector, 25)

In [ ]:
for hits in search_results:
    # get the distances to the query vector from all returned hits
    for hit, dis in zip(hits, hits.distances):
        # get the value of an output field specified in the search request.
        # dynamic fields are supported, but vector fields are not supported yet.
        print(hit.entity.get('uuid'))
        print(hit.entity.get('img_path'))
        print(dis)

# Face Detect Code from Script

In [ ]:
######## Facial Detection and Embedding  #########
        print(image_files_batch)
        resized_face_imgs = resize_imgs4face(resized_images)
        print('Detecting faces in images.')
        
        faces_aligned = []
        probs = []
        face_batch_size = 25
        number_batches = ceil(len(resized_face_imgs) / face_batch_size)
        
        for item in tqdm(chunk(resized_face_imgs,face_batch_size), desc='Face batchs:', total = number_batches):
            faces_aligned_i, probs_i = face_mtcnn.detect(item)
            faces_aligned.append(faces_aligned_i)
            probs.append(probs_i)

        # cannot do full batch, 25 is the max
        # faces_aligned, probs = face_mtcnn.detect(resized_face_imgs)
        
        if len(faces_aligned) > 0:
        
            # Sort through results
            # Construct list of ImageData instances
            # image_data_list = [
            #     ImageData(
            #         file, 
            #         item, 
            #         org_uuid, 
            #         p, 
            #         resized_images[i].shape[:2], # Assuming the shape is in the form (height, width, channels)
            #         resized_face_imgs[i].shape[:2] # Assuming the shape is in the form (height, width, channels)
            #     )
            #     for i, (file, box, prob, org_uuid) in enumerate(zip(image_files_batch, faces_aligned, probs, img_metadata['file_uuid'].tolist()))
            #     if isinstance(box, ndarray)
            #     for item, p in zip(box, prob)
            # ]
            
            # Flatten the results and include index to maintain reference to original image
            flat_results = [
                (i, face, prob, img_metadata['file_uuid'][i], image_files_batch[i])
                for i, (faces, probs) in enumerate(zip(faces_aligned, probs))
                for face, prob in zip(faces, probs) if face is not None
            ]
            
            print(flat_results)
            

            # Construct list of ImageData instances
            image_data_list = [
                ImageData(
                    file,
                    box[0],
                    uuid,
                    p[0],
                    resized_images[orig_index].shape[:2],  # Assuming the shape is in the form (height, width, channels)
                    resized_face_imgs[orig_index].shape[:2]  # Assuming the shape is in the form (height, width, channels)
                )
                for orig_index, box, p, uuid, file in flat_results
                if isinstance(box, ndarray) and box is not None
            ]
    
            items_dict = [asdict(item) for item in image_data_list]
            df = pd.DataFrame(items_dict)
            print(df)
            
            # Flatten the results and include index to maintain reference to original image
            # flat_results = [
            #     (i, face, prob, img_metadata['file_uuid'][i], image_files_batch[i])
            #     for i, (faces, probs) in enumerate(zip(faces_aligned, probs))
            #     if faces is not None
            #     for face, prob in zip(faces, probs)
            # ]

            # # Construct list of ImageData instances
            # image_data_list = [
            #     ImageData(
            #         file,
            #         box,
            #         uuid,
            #         p,
            #         resized_images[orig_index].shape[:2],  # Assuming the shape is in the form (height, width, channels)
            #         resized_face_imgs[orig_index].shape[:2]  # Assuming the shape is in the form (height, width, channels)
            #     )
            #     for orig_index, box, p, uuid, file in flat_results
            #     if isinstance(box, ndarray)
            # ]
            # # print(image_data_list)
            
            # Databasing face detections
            print('Databasing face detections')
            # Generate the records using parallel processing
            df_faces = parallel_convert_coordinates(image_data_list)
            print(df_faces)

            db_df_faces = df_faces.drop(columns=['path'])
            insert_detections_bulk(db_df_faces)
            
            # Prepare for face embeddings
            print('Preparing data for face embeddings')        
            extracted_faces = extract_faces_from_data(df_faces, device)
            embeddings = face_resnet(extracted_faces)
            # covnert to list
            vector_list = [normalize_to_L2(tensor.tolist()).tolist() for tensor in embeddings]
            
            # Database the face embeddings
            print('Databasing face embeddings')
            # write the crop embeddings to milvus
            facial_collection.insert([df_faces['uuid'].tolist(), df_faces['path'].tolist(), vector_list, list(repeat('face', len(df_faces))), list(repeat('none', len(df_faces)))])
            
            del df_faces, resized_face_imgs, faces_aligned, probs, embeddings
            collect()
            cuda.empty_cache()
            print('Memory cleared')
        else:
            print("No faces in this batch.")
            del faces_aligned, probs, resized_face_imgs
            collect()
            cuda.empty_cache()
            print('Memory cleared')
